<a href="https://colab.research.google.com/github/NandiniJaiswal05/Hybrid-Quantum-Classical-Antibiotic-Stewardship-Susceptibility-Framework/blob/sanvi/pushtogit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
!pip install -q rdkit

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator, SaltRemover
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [39]:
df = pd.read_csv('/content/02_ChEMBL_Drug_Bioactivity_Dataset.csv', sep=';', quotechar='"')

In [40]:
df.isnull().sum()

,0
Molecule ChEMBL ID,0
Molecule Name,1735
Molecule Max Phase,1872
Molecular Weight,0
#RO5 Violations,58
AlogP,58
Compound Key,0
Smiles,0
Standard Type,0
Standard Relation,118


In [41]:
df.isnull().sum() / len(df) * 100

,0
Molecule ChEMBL ID,0.000000
Molecule Name,84.060078
Molecule Max Phase,90.697674
Molecular Weight,0.000000
#RO5 Violations,2.810078
AlogP,2.810078
Compound Key,0.000000
Smiles,0.000000
Standard Type,0.000000
Standard Relation,5.717054


In [42]:
df.columns = df.columns.str.strip()
df = df.dropna(how='all', axis=1)

high_missing = ['Assay Cell Type', 'Cell ChEMBL ID', 'Molecule Max Phase', 'Molecule Name']
cols_to_drop = [
    'Molecule ChEMBL ID', 'Compound Key', 'Assay ChEMBL ID', 'Target ChEMBL ID',
    'Document ChEMBL ID', 'Source ID', 'Source Description', 'Assay Description',
    'BAO Format ID', 'BAO Label', 'Assay Parameters', 'Target Name',
    'Target Type', 'Comment', 'Potential Duplicate'
]
df = df.drop(columns=[col for col in high_missing + cols_to_drop if col in df.columns], errors='ignore')


if 'Standard Relation' in df.columns:
    df = df[df['Standard Relation'].astype(str).str.contains('=', na=False)]

if 'pChEMBL Value' in df.columns and 'Standard Value' in df.columns:
    valid_std_mask = df['pChEMBL Value'].isna() & (df['Standard Value'] > 0)
    df.loc[valid_std_mask, 'pChEMBL Value'] = -np.log10(
        df.loc[valid_std_mask, 'Standard Value'] * 1e-9
    )

df = df.dropna(subset=['pChEMBL Value', 'Smiles']).copy()


remover = SaltRemover.SaltRemover()

def clean_smiles(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is not None:
        mol = remover.StripMol(mol)
        return Chem.MolToSmiles(mol, canonical=True)
    return None

df['Clean_Smiles'] = df['Smiles'].apply(clean_smiles)
df = df.dropna(subset=['Clean_Smiles']).copy()


physchem_cols = ['Molecular Weight', '#RO5 Violations', 'AlogP']
valid_physchem = [c for c in physchem_cols if c in df.columns]

agg_dict = {'pChEMBL Value': 'mean'}
for col in valid_physchem:
    agg_dict[col] = 'first'

df_aggregated = df.groupby('Clean_Smiles', as_index=False).agg(agg_dict)


df_aggregated[valid_physchem] = df_aggregated[valid_physchem].fillna(df_aggregated[valid_physchem].median())


mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

def smiles_to_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        return mfpgen.GetCountFingerprintAsNumPy(mol)
    return np.zeros((1024,))

x_fps = np.array([smiles_to_fingerprint(s) for s in df_aggregated['Clean_Smiles']])
x_physchem = df_aggregated[valid_physchem].values
y = df_aggregated['pChEMBL Value'].values


X_physchem_train, X_physchem_test, x_fps_train, x_fps_test, y_train, y_test = train_test_split(
    x_physchem, x_fps, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_physchem_train_scaled = scaler.fit_transform(X_physchem_train)
X_physchem_test_scaled = scaler.transform(X_physchem_test)

X_train = np.hstack([X_physchem_train_scaled, x_fps_train])
X_test = np.hstack([X_physchem_test_scaled, x_fps_test])

print("--- Classical Preprocessing Complete ---")
print(f"Total Unique Compounds Kept: {len(df_aggregated)}")
print(f"X_train Shape: {X_train.shape}, y_train Shape: {y_train.shape}")
print(f"X_test Shape:  {X_test.shape}, y_test Shape:  {y_test.shape}")

--- Classical Preprocessing Complete ---
Total Unique Compounds Kept: 226
X_train Shape: (180, 1027), y_train Shape: (180,)
X_test Shape:  (46, 1027), y_test Shape:  (46,)


In [43]:
feature_names = valid_physchem + [f'FP_Bit_{i}' for i in range(x_fps.shape[1])]
df_X_train = pd.DataFrame(X_train, columns=feature_names)

print("--- Top 5 Rows of Training Features (X_train) ---")
display(df_X_train.head())

print("\n--- Top 5 Training Targets (y_train: pChEMBL Value) ---")
print(y_train[:5])

print("\n--- Top 5 Aggregated Base DataFrame Rows ---")
display(df_aggregated[['Clean_Smiles'] + valid_physchem + ['pChEMBL Value']].head())

--- Top 5 Rows of Training Features (X_train) ---


,Molecular Weight,#RO5 Violations,AlogP,FP_Bit_0,FP_Bit_1,FP_Bit_2,FP_Bit_3,FP_Bit_4,FP_Bit_5,FP_Bit_6,...,FP_Bit_1014,FP_Bit_1015,FP_Bit_1016,FP_Bit_1017,FP_Bit_1018,FP_Bit_1019,FP_Bit_1020,FP_Bit_1021,FP_Bit_1022,FP_Bit_1023
0,0.277144,-0.498714,0.691618,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.211767,1.541479,1.664407,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,-0.709644,-0.498714,0.898228,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.517931,-0.498714,0.691618,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.165429,1.541479,0.011526,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



--- Top 5 Training Targets (y_train: pChEMBL Value) ---
[6.84769383 7.90787864 7.10064668 8.04067905 8.05792888]

--- Top 5 Aggregated Base DataFrame Rows ---


,Clean_Smiles,Molecular Weight,#RO5 Violations,AlogP,pChEMBL Value
0,CC(C)(C)c1cc(C=C(C#N)C#N)cc(Br)c1O,305.17,0.0,3.88,7.027760
1,CC(C)CN(C)C(=O)c1n[nH]c2cc(O)c(C(=O)N(C)c3ccc(...,465.55,0.0,3.11,7.909386
2,CC(C)N(C)S(=O)(=O)c1cc(-c2n[nH]c(=O)n2-c2ccccc...,438.89,0.0,2.32,6.011548
3,CC(C)N(C)S(=O)(=O)c1cc(-c2n[nH]c(=O)n2-c2ccccc...,422.44,0.0,1.81,6.333816
4,CC(C)N1CCC(NC(=O)c2cc3ccccc3n2CC(=O)Nc2ccc(Br)...,497.44,0.0,4.65,7.784975
